# Project: Bayesian A/B test with MCMC 🧪
Two versions of your study-note template: did version B really make you finish more exercises? Answer the question you actually care about, *"what's the probability B is better, and by how much?"*, with a Bayesian model sampled by a **Metropolis–Hastings** sampler you write yourself, and check it against the exact Beta posterior.

Topic: [[MCMC]], [[Bayesian Inference]] · guide note: [[Project - Bayesian AB Test with MCMC]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
nA, kA = 120, 54        # version A: 54 of 120 exercise sessions completed
nB, kB = 115, 66        # version B: 66 of 115
print(f"observed rates: A {kA/nA:.1%}, B {kB/nB:.1%}")

## 1. Log posterior\nUniform priors on $p_A, p_B \in (0,1)$ and binomial likelihoods. Return the log posterior up to a constant: $k\log p + (n-k)\log(1-p)$ summed over both versions, and $-\infty$ outside $(0,1)$.

In [ ]:
def log_post(pA, pB):
    # TODO 1: log-likelihood of both groups (flat prior), -inf outside (0, 1)
    raise NotImplementedError  # TODO 1

## 2. Random-walk Metropolis\nPropose $x' = x + \mathcal N(0, \text{step}^2 I)$ and accept with probability $\min(1, e^{\log p(x') - \log p(x)})$; otherwise stay (and record the current point again). Return `(samples, acceptance_rate)`.

In [ ]:
def metropolis(log_post, start, n=20000, step=0.05, seed=0):
    # TODO 2: random-walk Metropolis–Hastings
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_mh():
    s, acc = metropolis(lambda a: -0.5 * a ** 2 if np.isscalar(a) else -0.5 * float(np.sum(np.asarray(a) ** 2)), [0.0], n=20000, step=1.0)
    lp_ok = np.isinf(log_post(1.2, 0.5)) and np.isclose(log_post(0.5, 0.5), (nA + nB) * np.log(0.5))
    return lp_ok and abs(s[2000:].mean()) < 0.1 and abs(s[2000:].std() - 1) < 0.1 and 0.3 < acc < 0.9
check("metropolis", _t_mh, "On a standard normal target the samples must have mean ≈ 0 and std ≈ 1; log_post must be −inf outside (0,1).")
if ready("metropolis"):
    samples, acc = metropolis(log_post, [0.5, 0.5]); s = samples[2000:]               # drop burn-in
    pA, pB = s[:, 0], s[:, 1]; prob_b = np.mean(pB > pA); lift = pB - pA
    exact = np.mean(stats.beta(1 + kB, 1 + nB - kB).rvs(200000, random_state=1) > stats.beta(1 + kA, 1 + nA - kA).rvs(200000, random_state=2))
    print(f"acceptance {acc:.0%}; P(B better) MCMC {prob_b:.3f} vs exact {exact:.3f}; lift 95 % interval [{np.quantile(lift, .025):+.3f}, {np.quantile(lift, .975):+.3f}]")
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
    ax[0].plot(samples[:3000, 0], lw=.5, label="pA"); ax[0].plot(samples[:3000, 1], lw=.5, label="pB"); ax[0].set_title("trace (first 3000)"); ax[0].legend()
    ax[1].hist(pA, 60, alpha=.6, density=True, label="A"); ax[1].hist(pB, 60, alpha=.6, density=True, label="B"); ax[1].legend(); ax[1].set_title("posteriors")
    ax[2].hist(lift, 60, density=True); ax[2].axvline(0, c="k"); ax[2].set_title("pB − pA"); plt.show()
    check("matches exact posterior", lambda: abs(prob_b - exact) < 0.02, "MCMC's P(B > A) should be within 0.02 of the exact Beta answer.")

<details><summary>▶ Solution</summary>

```python
def log_post(pA, pB):
    if not (0 < pA < 1 and 0 < pB < 1):
        return -np.inf
    return (kA * np.log(pA) + (nA - kA) * np.log(1 - pA) +
            kB * np.log(pB) + (nB - kB) * np.log(1 - pB))
```

```python
def metropolis(log_post, start, n=20000, step=0.05, seed=0):
    rng = np.random.default_rng(seed)
    x = np.array(start, float); lp = log_post(*x)
    samples, accepted = [], 0
    for _ in range(n):
        prop = x + rng.normal(0, step, size=len(x))
        lp_prop = log_post(*prop)
        if np.log(rng.random()) < lp_prop - lp:
            x, lp = prop, lp_prop; accepted += 1
        samples.append(x.copy())
    return np.array(samples), accepted / n
```
</details>

## Stretch goals
- Step size: try 0.001 and 0.5. Look at the traces and acceptance rates. What's "mixing"?
- Compute the effective sample size (autocorrelation) of your chain.
- Compare with a frequentist two-proportion z-test ([[ML Formula Sheet]] §18): same data, different question.